# Foxworks PicoRV32 bring-up on the AUP-ZU3

This notebook plays the role the RP2040 firmware plays on the real Tiny Tapeout demo board: load the flash image, sequence `ena`/`rst_n`, then watch the chip run.

Order of operations mirrors demo-board bring-up exactly: **chip in reset → program flash → verify → ena → release reset → UART/GPIO out**.

Prerequisites: `tt_foxworks_picorv32.bit` + `tt_foxworks_picorv32.hwh` (from `zu3/build_tt_foxworks_picorv32.tcl`) and `flash.bin` (from `make -C fw`) copied next to this notebook.

In [ ]:
from pynq import Overlay, MMIO
import numpy as np, time

ol = Overlay('tt_foxworks_picorv32.bit')
print(sorted(ol.ip_dict.keys()))  # expect: ctrl_gpio, uartlite, vdb (+ PS blocks)


In [ ]:
# MMIO handles. AXI GPIO register map: ch1 data @0x00, ch2 data @0x08.
vdb  = MMIO(ol.ip_dict['vdb']['phys_addr'], 0x10000)   # 64 KB flash window
gpio = MMIO(ol.ip_dict['ctrl_gpio']['phys_addr'], 0x1000)
uart = MMIO(ol.ip_dict['uartlite']['phys_addr'], 0x1000)

CTRL = 0x00   # ch1: bit0 = rst_n, bit1 = ena
UO   = 0x08   # ch2: live uo_out[7:0] readback

gpio.write(CTRL, 0b00)  # chip in reset: uio bus released to the flash model
print('chip held in reset')


In [ ]:
from pynq.ps import Clocks
print('pl_clk0 (chip):', Clocks.fclk0_mhz)
print('pl_clk1 (uartlite):', Clocks.fclk1_mhz)

In [ ]:
# Program the 64 KB flash (image already padded: reset vector at 0x400)
fw = np.fromfile('flash.bin', dtype=np.uint32)
print(f'{fw.nbytes} bytes to load')
try:
    vdb.array[:len(fw)] = fw                    # fast path
except Exception as e:
    print('mmio.array unavailable, word loop:', e) # Sometimes even if mmio is available it fails
    for i, w in enumerate(fw):
        vdb.write(4 * i, int(w))
print('flash programmed')


In [ ]:
# Verify (spot-check readback). Only valid while the SoC is in reset -
# the loader shares its BRAM read port with the SPI prefetcher.
bad = [i for i in range(0, len(fw), 64)
       if (vdb.read(4 * i) & 0xFFFFFFFF) != int(fw[i])]
print('verify OK' if not bad else f'MISMATCH at words {bad[:8]}')


In [ ]:
# Demo-board power-on sequence: ena first, then release reset.
gpio.write(CTRL, 0b10)  # ena=1, still in reset
gpio.write(CTRL, 0b11)  # rst_n=1 - the SoC boots from emulated flash
print('running')


In [ ]:
# uartlite's RX FIFO is 16 bytes = 1.4 ms at 115200. Any sleep longer
# than that while data streams LOSES BYTES (symptom: 16-char shards).
# So: busy-poll while data flows (~1 us/MMIO read, lossless), rest
# only once the line has been idle.
t0, last, buf = time.time(), time.time(), ''
while time.time() - t0 < 20:
    got = False
    while uart.read(0x08) & 0x1:
        buf += chr(uart.read(0x00) & 0xFF)
        got = True
    if got:
        last = time.time()
    elif time.time() - last > 2.0:   # 2 s of silence = report finished
        break
print(buf if buf else '(nothing received - check reset sequencing / clkdiv)')


# Bounce Test
Displays the Output pins as a block bouncing left to right

Build the self-test firmware (`make -C fw clean && make -C fw PROG=bounce`), copy the new `flash.bin` here, re-run the program/verify/release cells above, then run the two cells below.

In [ ]:
# Live bounce viewer - reads uo_out through AXI GPIO ch2 and updates
# a display handle in place.
from IPython.display import display, HTML
h = display(HTML(''), display_id=True)
for _ in range(600):
    v = gpio.read(UO) & 0xFF
    row = ''.join('\u2588' if (v >> i) & 1 else '\u00b7' for i in range(8))
    h.update(HTML(f'<pre style="font-size:36px;letter-spacing:8px">{row}</pre>'))
    time.sleep(0.05)


## Core self-test

Build the self-test firmware (`make -C fw clean && make -C fw PROG=selftest`), copy the new `flash.bin` here, re-run the program/verify/release cells above, then run the two cells below.

The firmware exercises the ALU, the serial shifter, signed/unsigned compares, SRAM byte lanes, XIP data loads, libgcc soft mul/div, stack discipline, and the non-sequential-fetch (CS-pause) path - then reports every result over UART, posts a verdict on `uo_out` (0xC3 pass / 0x3C fail), and drops into an echo service (every RX byte returned as byte+1) so the UART receive path can be verified interactively.


The report is re-requestable at any time by sending `?` over the UART - the parse cell does this automatically, so it can be run (and re-run) whenever.

In [ ]:
# Request and parse the self-test report. '?' makes the report
# on-demand (16-byte RX FIFO makes the boot copy perishable), and the
# DRAIN DISCIPLINE below makes collection lossless: busy-poll while
# data flows - a sleep anywhere in the hot path drops ~230 chars per
# 20 ms against a 16-byte FIFO (symptom: 16-char shards of report).
import re, time
uart.write(0x0C, 0x02)          # reset RX FIFO: drop stale fragments
uart.write(0x04, ord('?'))      # request a fresh run + report
t0, last, buf = time.time(), time.time(), ''
while time.time() - t0 < 30 and 'ECHO-SERVICE' not in buf:
    got = False
    while uart.read(0x08) & 0x1:
        buf += chr(uart.read(0x00) & 0xFF)
        got = True
    if got:
        last = time.time()
    elif time.time() - last > 3.0:
        break
print(buf)
tests = dict(re.findall(r'T(\d+) \S+ (PASS|FAIL)', buf))
result = re.search(r'RESULT (\d+)/(\d+) (PASS|FAIL)', buf)
verdict = gpio.read(UO) & 0xFF
uart_ok = bool(result) and result.group(3) == 'PASS' and 'FAIL' not in tests.values()
pin_ok  = verdict == 0xC3
print(f'UART report: {"PASS" if uart_ok else "FAIL"} ({len(tests)} tests parsed)')
print(f'uo_out verdict: 0x{verdict:02x} ({"PASS" if pin_ok else "FAIL/unexpected"})')
if 0x10 <= verdict <= 0x1c:
    print(f'-> frozen PROGRESS code: core hung during T{verdict-0x10:02d} '
          '(no report + frozen progress usually means stack overflow '
          'or a wedged bus access in that test)')
assert uart_ok and pin_ok, 'SELF-TEST FAILED - see report above'
print('CORE SELF-TEST: PASS (report and pins agree)')


In [ ]:
# Interactive UART RX verification via the echo service (byte -> byte+1).
# uartlite: TX FIFO @0x04, status @0x08 (bit0 rx-valid, bit3 tx-full)
challenge = b'PYNQ-echo-check'
rx = bytearray()
for c in challenge:
    while uart.read(0x08) & 0x8:
        pass
    uart.write(0x04, c)
t0 = time.time()
while len(rx) < len(challenge) and time.time() - t0 < 3:
    while uart.read(0x08) & 0x1:
        rx.append(uart.read(0x00) & 0xFF)
    time.sleep(0.001)
expect = bytes((c + 1) & 0xFF for c in challenge)
print(f'sent    : {challenge}')
print(f'received: {bytes(rx)}')
assert bytes(rx) == expect, 'echo mismatch - UART RX path broken'
print('UART RX PATH: PASS (echo transform verified)')


In [ ]:
# Halt the chip (back into reset; flash contents persist, so a
# re-release reruns the same image without reprogramming).
gpio.write(CTRL, 0b00)
print('chip in reset')
